# 📄 배터리 레포트 생성 이후 문서 비교 Agent Tutorial 

이 노트북은 앞선 **배터리 레포트 생성 에이전트** 이후에 바로 이어지는 2편입니다.

이번 버전에서는 문서 비교 Agent의 흐름을 아래처럼 명확히 정의합니다.

## 최종 파이프라인

1. **1편에서 생성된 DOCX 2개**를 입력받습니다.  
2. 각 DOCX에서 필요한 핵심 변수값은 **LLM이 JSON 스키마 형태로 추출**합니다.  
3. LLM 응답에서 **JSON을 파싱**합니다.  
4. 기준 문서와 비교 문서의 값 비교는 **하드코딩된 Python 로직**으로 수행합니다.  
5. 코드가 비교 결과를 **텍스트 보고용 요약 문자열**로 만듭니다.  
6. 그 비교 텍스트를 다시 **LLM에 넣어 최종 비교 레포트**를 생성합니다.  

즉, 이번 구조는 아래와 같습니다.

- **값 추출은 LLM**
- **값 비교는 코드**
- **최종 해석/보고서는 다시 LLM**

이 구조가 필요한 이유는 문서 형식이 조금 변해도 LLM이 비교적 유연하게 핵심 값을 뽑을 수 있고,  
실제 비교 기준은 코드로 고정하여 일관성을 유지할 수 있기 때문입니다.


## 💡 목차

> (1) 1편 레포트 생성 Agent와 연결 구조  
> (2) 이번 비교 Agent의 정확한 역할  
> (3) 입력 문서 준비  
> (4) DOCX 텍스트 읽기  
> (5) LLM으로 핵심 변수 JSON 추출  
> (6) JSON 파싱 및 정규화  
> (7) 하드코딩 비교 로직 작성  
> (8) 비교 결과를 텍스트로 변환  
> (9) LLM으로 최종 비교 레포트 생성  
> (10) 결과 DOCX 저장  
> (11) 전체 함수 리팩토링  


## (1) 1편 레포트 생성 Agent와 연결 구조

앞선 1편에서는 다음 과정을 수행했습니다.

- CSV를 읽어 배터리 데이터를 분석
- 전압, 전류, 온도 그래프를 생성
- 통계값을 계산
- DOCX 평가 레포트를 자동 생성

따라서 이번 2편의 입력은 **CSV가 아니라 1편에서 생성된 DOCX 레포트**입니다.

이번 2편은 1편 레포트 내부에 이미 기록된 다음 값들을 다시 읽어와 비교하는 구조입니다.

- 파일명
- 데이터 수
- 최대 전압
- 최소 전압
- 최대 온도
- 최소 온도
- 모듈 상태
- 선택적으로 LLM 요약
- 선택적으로 LLM 판정


## (2) 이번 비교 Agent의 정확한 역할

이번 Agent는 단순한 문서 diff 도구가 아닙니다.

이번 Agent는 **문서 안에 들어 있는 구조화 가능한 변수값을 LLM으로 추출한 뒤**,  
그 값을 코드로 비교하고,  
비교 결과만 다시 LLM에게 해석하게 하는 구조입니다.

정리하면 역할은 아래와 같습니다.

- **LLM 1차 역할**: DOCX 텍스트에서 핵심 변수값 추출
- **코드 역할**: 기준 문서와 비교 문서 값 비교
- **LLM 2차 역할**: 비교 결과를 사람이 읽기 좋은 보고서로 생성


## (3) 입력 문서 준비

이번 예제에서는 1편에서 생성된 DOCX를 그대로 입력으로 사용합니다.


In [2]:
# 설치가 필요하면 주석을 해제하세요.
# !pip install python-docx

import json
import re
from pathlib import Path
from typing import Any
from docx import Document

In [3]:
baseline_path = Path("./Result/battery_report_1000.docx")
compare_path = Path("./Result/battery_report_1001.docx")
output_compare_report = Path("./Result/compare_report.docx")

print("기준 문서:", baseline_path)
print("비교 문서:", compare_path)
print("비교 결과 문서:", output_compare_report)

기준 문서: Result/battery_report_1000.docx
비교 문서: Result/battery_report_1001.docx
비교 결과 문서: Result/compare_report.docx


## (4) DOCX 텍스트 읽기

먼저 DOCX 문서 전체 텍스트를 읽어옵니다.  
이번 단계에서는 **값을 직접 파싱하지 않습니다.**  
단지 LLM에 전달할 원문 텍스트를 준비합니다.


In [4]:
def read_docx_text(docx_path: str | Path) -> str:
    doc = Document(str(docx_path))
    lines = []
    for p in doc.paragraphs:
        text = p.text.strip()
        if text:
            lines.append(text)
    return "\n".join(lines)

In [5]:
if baseline_path.exists():
    baseline_text_preview = read_docx_text(baseline_path)
    print(baseline_text_preview[:1000])
else:
    print("기준 문서가 아직 없습니다.")

배터리 충방전 시험 데이터 분석 레포트
그래프 및 텍스트 자동 치환용 DOCX 템플릿
요약
1. 용량
그래프
분석 내용
2. 용접
그래프
분석 내용
3. 센서
그래프
분석 내용
4. 원시 데이터 요약
요약
본 배터리 충방전 시험 데이터 분석 레포트는 배터리 용량, 용접, 센서 성능을 평가합니다. 

**1. 용량:** 그래프 및 분석 내용은 배터리의 충방전 용량 변화를 보여주며, 전반적인 용량 유지율 및 성능 추세를 파악하는 데 사용됩니다.  

**2. 용접:** 용접부의 상태를 평가하기 위한 데이터를 포함합니다. 그래프와 분석 내용은 용접 불량 여부, 강도 등을 판단하는 데 활용됩니다.

**3. 센서:** 배터리 내부 센서의 측정값과 정상 범위를 비교하여 센서의 정확성 및 이상 유무를 검증합니다. 

**4. 원시 데이터 요약:** 시험 과정에서 수집된 모든 원시 데이터를 요약하여 제공함으로써, 세부적인 데이터 분석 및 추가 검토를 용이하게 합니다.

전반적으로, 이 보고서는 배터리의 전기적 성능, 물리적 결합 상태, 내부 감지 기능 등을 종합적으로 검토하여 배터리의 품질과 신뢰성을 평가하는 데 목적이 있습니다.


## (5) LLM으로 핵심 변수 JSON 추출

이번 단계가 핵심입니다.

문서의 텍스트 전체를 LLM에 넣고, 아래와 같은 **JSON 스키마**로 값을 추출하도록 요청합니다.

### 추출 대상 예시
- generated_at
- source_file_name
- row_count
- voltage_max
- voltage_min
- temperature_max
- temperature_min
- module_status
- llm_summary
- llm_verdict

이때 중요한 점은,  
LLM이 자유 서술을 하지 않도록 **반드시 JSON만 출력하도록 강하게 제한**하는 것입니다.


In [6]:
from google import genai
from utils.api import *

def invoke_llm(contents):
    API_KEY = gemini_api_key ## gemini api key 발급받아서 넣기 
    client = genai.Client(api_key=API_KEY)
    response = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=contents
    )
    return response.text
    

In [7]:
def extract_report_fields_with_llm(
    docx_text: str,
    document_name: str,
    invoke_llm_fn,
) -> str:
    prompt = f"""
다음은 배터리 평가 DOCX 레포트의 텍스트입니다.
이 문서에서 핵심 변수값을 추출하여 반드시 JSON만 출력하세요.
설명문, 마크다운, 코드블록은 절대 출력하지 마세요.

[문서명]
{document_name}

[문서 텍스트]
{docx_text}

반드시 아래 스키마를 따르세요.

{{
  "document_name": "{document_name}",
  "generated_at": null,
  "source_file_name": null,
  "row_count": null,
  "voltage_max": null,
  "voltage_min": null,
  "temperature_max": null,
  "temperature_min": null,
  "module_status": null,
  "llm_summary": null,
  "llm_verdict": null
}}

규칙:
- 값을 찾을 수 없으면 null
- 숫자는 숫자형으로 출력
- 문장 요약은 문자열
- JSON 외의 다른 텍스트 금지
"""
    return invoke_llm_fn(prompt)

## (6) JSON 파싱 및 정규화

LLM이 반환한 JSON 문자열을 안전하게 파싱하고,  
숫자형 값은 숫자로 정규화합니다.

또한 비교 편의를 위해 파생값도 계산합니다.

- 전압 범위 = 최대 전압 - 최소 전압
- 온도 범위 = 최대 온도 - 최소 온도


In [8]:
def parse_llm_json(text: str) -> dict[str, Any]:
    text = text.strip()

    fenced = re.search(r"```json\s*(\{.*?\})\s*```", text, re.DOTALL)
    if fenced:
        return json.loads(fenced.group(1))

    plain = re.search(r"(\{.*\})", text, re.DOTALL)
    if plain:
        return json.loads(plain.group(1))

    raise ValueError("LLM 응답에서 JSON을 찾지 못했습니다.")


def to_number_if_possible(value):
    if value is None:
        return None
    if isinstance(value, (int, float)):
        return value

    text = str(value).strip().replace(",", "")
    if text == "":
        return None

    try:
        num = float(text)
        if abs(num - int(num)) < 1e-12:
            return int(num)
        return num
    except Exception:
        return value


def normalize_extracted_fields(data: dict[str, Any]) -> dict[str, Any]:
    normalized = dict(data)

    numeric_fields = [
        "row_count",
        "voltage_max",
        "voltage_min",
        "temperature_max",
        "temperature_min",
    ]

    for field in numeric_fields:
        normalized[field] = to_number_if_possible(normalized.get(field))

    v_max = normalized.get("voltage_max")
    v_min = normalized.get("voltage_min")
    t_max = normalized.get("temperature_max")
    t_min = normalized.get("temperature_min")

    normalized["voltage_range"] = None
    normalized["temperature_range"] = None

    if isinstance(v_max, (int, float)) and isinstance(v_min, (int, float)):
        normalized["voltage_range"] = round(float(v_max) - float(v_min), 6)

    if isinstance(t_max, (int, float)) and isinstance(t_min, (int, float)):
        normalized["temperature_range"] = round(float(t_max) - float(t_min), 6)

    return normalized

In [ ]:
# 실행 예시
# if baseline_path.exists():
#     baseline_text = read_docx_text(baseline_path)
#     baseline_raw_json = extract_report_fields_with_llm(
#         docx_text=baseline_text,
#         document_name=baseline_path.name,
#         invoke_llm_fn=invoke_llm
#     )
#     baseline_fields = normalize_extracted_fields(parse_llm_json(baseline_raw_json))
#     print(json.dumps(baseline_fields, ensure_ascii=False, indent=2, default=str))

## (7) 하드코딩 비교 로직 작성

이제 비교는 LLM이 아니라 **코드**가 수행합니다.

비교 대상은 아래 세 종류로 나눕니다.

### 수치형 필드
- row_count
- voltage_max
- voltage_min
- voltage_range
- temperature_max
- temperature_min
- temperature_range

### 범주형 필드
- module_status
- llm_verdict

### 텍스트형 필드
- source_file_name
- llm_summary

이렇게 하면 비교 기준이 항상 일관되며,  
LLM의 변동성 없이 안정적으로 차이를 계산할 수 있습니다.


In [9]:
NUMERIC_FIELDS = [
    "row_count",
    "voltage_max",
    "voltage_min",
    "voltage_range",
    "temperature_max",
    "temperature_min",
    "temperature_range",
]

CATEGORICAL_FIELDS = [
    "module_status",
    "llm_verdict",
]

TEXT_FIELDS = [
    "source_file_name",
    "llm_summary",
]


def compare_report_fields_hardcoded(
    baseline_fields: dict[str, Any],
    compare_fields: dict[str, Any],
) -> dict[str, Any]:
    changes = []

    def add_change(field_name, field_type, before, after):
        if before == after:
            return

        item = {
            "field": field_name,
            "field_type": field_type,
            "before": before,
            "after": after,
            "delta": None,
        }

        if field_type == "numeric":
            if isinstance(before, (int, float)) and isinstance(after, (int, float)):
                item["delta"] = round(float(after) - float(before), 6)

        changes.append(item)

    for field in NUMERIC_FIELDS:
        add_change(field, "numeric", baseline_fields.get(field), compare_fields.get(field))

    for field in CATEGORICAL_FIELDS:
        add_change(field, "categorical", baseline_fields.get(field), compare_fields.get(field))

    for field in TEXT_FIELDS:
        add_change(field, "text", baseline_fields.get(field), compare_fields.get(field))

    return {
        "baseline_document": baseline_fields.get("document_name"),
        "compare_document": compare_fields.get("document_name"),
        "total_changed_fields": len(changes),
        "numeric_change_count": sum(1 for x in changes if x["field_type"] == "numeric"),
        "categorical_change_count": sum(1 for x in changes if x["field_type"] == "categorical"),
        "text_change_count": sum(1 for x in changes if x["field_type"] == "text"),
        "changes": changes,
    }

## (8) 비교 결과를 텍스트로 변환

코드가 비교한 결과를 사람이 읽을 수 있는 **텍스트 요약**으로 변환합니다.

이 텍스트는 다음 단계에서 다시 LLM에 입력되어  
최종 비교 레포트의 재료로 사용됩니다.

즉, 이번 구조는 아래와 같습니다.

- LLM이 값 추출
- 코드가 비교
- 코드가 비교 결과 텍스트 생성
- 그 텍스트를 다시 LLM이 최종 보고서화


In [10]:
def build_comparison_text(
    baseline_fields: dict[str, Any],
    compare_fields: dict[str, Any],
    compare_result: dict[str, Any],
) -> str:
    lines = []
    lines.append(f"기준 문서: {compare_result['baseline_document']}")
    lines.append(f"비교 문서: {compare_result['compare_document']}")
    lines.append(f"전체 변경 필드 수: {compare_result['total_changed_fields']}")
    lines.append(f"수치형 변경 수: {compare_result['numeric_change_count']}")
    lines.append(f"범주형 변경 수: {compare_result['categorical_change_count']}")
    lines.append(f"텍스트형 변경 수: {compare_result['text_change_count']}")
    lines.append("")

    if not compare_result["changes"]:
        lines.append("변경된 필드가 없습니다.")
        return "\n".join(lines)

    lines.append("[상세 변경 내역]")
    for item in compare_result["changes"]:
        lines.append(f"- 필드: {item['field']} ({item['field_type']})")
        lines.append(f"  기준값: {item['before']}")
        lines.append(f"  비교값: {item['after']}")
        if item.get("delta") is not None:
            lines.append(f"  변화량: {item['delta']}")
        lines.append("")

    return "\n".join(lines)

In [11]:
# 비교 텍스트 예시
if baseline_path.exists() and compare_path.exists():
    baseline_text = read_docx_text(baseline_path)
    compare_text = read_docx_text(compare_path)

    baseline_raw_json = extract_report_fields_with_llm(baseline_text, baseline_path.name, invoke_llm)
    compare_raw_json = extract_report_fields_with_llm(compare_text, compare_path.name, invoke_llm)

    baseline_fields = normalize_extracted_fields(parse_llm_json(baseline_raw_json))
    compare_fields = normalize_extracted_fields(parse_llm_json(compare_raw_json))

    compare_result = compare_report_fields_hardcoded(baseline_fields, compare_fields)
    compare_text_result = build_comparison_text(baseline_fields, compare_fields, compare_result)
    print(compare_text_result)

기준 문서: battery_report_1000.docx
비교 문서: battery_report_1001.docx
전체 변경 필드 수: 2
수치형 변경 수: 0
범주형 변경 수: 1
텍스트형 변경 수: 1

[상세 변경 내역]
- 필드: llm_verdict (categorical)
  기준값: None
  비교값: Pass

- 필드: llm_summary (text)
  기준값: 본 배터리 충방전 시험 데이터 분석 레포트는 배터리 용량, 용접, 센서 성능을 평가합니다. 그래프 및 분석 내용은 배터리의 충방전 용량 변화, 용접부 상태, 센서의 측정값과 정상 범위를 비교하여 배터리의 품질과 신뢰성을 종합적으로 검토합니다.
  비교값: 본 배터리 충방전 시험 데이터 분석 보고서는 배터리의 용량, 용접, 센서 성능을 평가하는 데 중점을 두었습니다. 각 항목별로 제시된 그래프와 분석 내용을 종합적으로 검토한 결과, 배터리의 전반적인 성능은 양호한 것으로 판단됩니다. 특히, 용량 측정 결과는 설계 사양을 만족하는 수준으로 나타났으며, 용접 상태 또한 안정적인 것으로 평가되었습니다. 센서 데이터 역시 정상 범위 내에서 작동함을 확인했습니다. 원시 데이터 요약 정보는 이러한 분석 결과의 근거를 제공합니다.



## (9) LLM으로 최종 비교 레포트 생성

이제 마지막으로, 코드가 만든 비교 결과 텍스트를 LLM에 전달하여  
사람이 읽기 좋은 최종 비교 레포트를 생성합니다.

여기서 LLM의 역할은 다시 **해석과 요약**입니다.  
즉, 추출이나 비교 자체는 이미 끝난 상태이고,  
LLM은 그 결과를 **품질 관점 / 위험도 관점 / 의미 해석**으로 정리합니다.


In [12]:
def generate_final_comparison_report_with_llm(
    baseline_fields: dict[str, Any],
    compare_fields: dict[str, Any],
    comparison_text: str,
    invoke_llm_fn,
) -> dict[str, Any]:
    prompt = f"""
다음은 배터리 평가 레포트 두 개에서 추출한 핵심 변수값과,
코드 기반 비교 결과 텍스트입니다.

이 정보를 바탕으로 최종 비교 레포트를 작성하세요.
반드시 JSON만 출력하세요.

[기준 문서 추출값]
{json.dumps(baseline_fields, ensure_ascii=False, indent=2, default=str)}

[비교 문서 추출값]
{json.dumps(compare_fields, ensure_ascii=False, indent=2, default=str)}

[비교 결과 텍스트]
{comparison_text}

반드시 아래 JSON 형식만 출력하세요.

{{
  "comparison_summary": "전체 비교 요약",
  "risk_assessment": "위험도 또는 품질 관점 해석",
  "important_changes": [
    {{
      "field": "필드명",
      "reason": "왜 중요한지"
    }}
  ],
  "recommended_action": "실무적으로 어떤 조치를 권장하는지",
  "conclusion": "최종 결론"
}}
"""

    llm_response = invoke_llm_fn(prompt)
    return parse_llm_json(llm_response)

## (10) 결과 DOCX 저장

최종적으로는 아래 정보들을 모아 비교 결과 DOCX를 생성합니다.

- 기준 문서 / 비교 문서 이름
- 변경 통계
- 코드 비교 결과 텍스트
- LLM 최종 비교 레포트


In [13]:
def write_compare_report_docx(
    output_path: str | Path,
    compare_result: dict[str, Any],
    comparison_text: str,
    llm_final_report: dict[str, Any] | None = None,
):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    doc = Document()
    doc.add_heading("배터리 평가 레포트 비교 결과", 0)

    doc.add_heading("1. 비교 대상", level=1)
    doc.add_paragraph(f"기준 문서: {compare_result['baseline_document']}")
    doc.add_paragraph(f"비교 문서: {compare_result['compare_document']}")

    doc.add_heading("2. 변경 통계", level=1)
    doc.add_paragraph(f"전체 변경 필드 수: {compare_result['total_changed_fields']}")
    doc.add_paragraph(f"수치형 변경 수: {compare_result['numeric_change_count']}")
    doc.add_paragraph(f"범주형 변경 수: {compare_result['categorical_change_count']}")
    doc.add_paragraph(f"텍스트형 변경 수: {compare_result['text_change_count']}")

    doc.add_heading("3. 코드 기반 비교 결과", level=1)
    doc.add_paragraph(comparison_text)

    if llm_final_report is not None:
        doc.add_heading("4. LLM 최종 비교 레포트", level=1)
        doc.add_paragraph(f"비교 요약: {llm_final_report.get('comparison_summary')}")
        doc.add_paragraph(f"위험도 해석: {llm_final_report.get('risk_assessment')}")
        doc.add_paragraph(f"권장 조치: {llm_final_report.get('recommended_action')}")
        doc.add_paragraph(f"결론: {llm_final_report.get('conclusion')}")

        important_changes = llm_final_report.get("important_changes", [])
        if important_changes:
            doc.add_paragraph("주요 변경 항목")
            for item in important_changes:
                doc.add_paragraph(f"- {item.get('field')}: {item.get('reason')}")

    doc.save(str(output_path))
    return str(output_path)

## (11) 전체 함수 리팩토링

이제 전체 흐름을 하나의 함수로 묶습니다.

### 전체 파이프라인
1. DOCX 텍스트 읽기
2. LLM으로 값 JSON 추출
3. JSON 파싱 및 정규화
4. 코드로 하드코딩 비교
5. 비교 결과 텍스트 생성
6. LLM으로 최종 비교 레포트 생성
7. DOCX로 저장


In [14]:
def create_report_comparison_result_v2(
    baseline_docx_path: str | Path,
    compare_docx_path: str | Path,
    output_docx_path: str | Path,
    invoke_llm_fn,
):
    baseline_docx_path = Path(baseline_docx_path)
    compare_docx_path = Path(compare_docx_path)

    baseline_text = read_docx_text(baseline_docx_path)
    compare_text = read_docx_text(compare_docx_path)

    baseline_raw_json = extract_report_fields_with_llm(
        docx_text=baseline_text,
        document_name=baseline_docx_path.name,
        invoke_llm_fn=invoke_llm_fn,
    )
    compare_raw_json = extract_report_fields_with_llm(
        docx_text=compare_text,
        document_name=compare_docx_path.name,
        invoke_llm_fn=invoke_llm_fn,
    )

    baseline_fields = normalize_extracted_fields(parse_llm_json(baseline_raw_json))
    compare_fields = normalize_extracted_fields(parse_llm_json(compare_raw_json))

    compare_result = compare_report_fields_hardcoded(baseline_fields, compare_fields)
    comparison_text = build_comparison_text(baseline_fields, compare_fields, compare_result)

    llm_final_report = generate_final_comparison_report_with_llm(
        baseline_fields=baseline_fields,
        compare_fields=compare_fields,
        comparison_text=comparison_text,
        invoke_llm_fn=invoke_llm_fn,
    )

    saved_path = write_compare_report_docx(
        output_path=output_docx_path,
        compare_result=compare_result,
        comparison_text=comparison_text,
        llm_final_report=llm_final_report,
    )

    return {
        "baseline_raw_json": baseline_raw_json,
        "compare_raw_json": compare_raw_json,
        "baseline_fields": baseline_fields,
        "compare_fields": compare_fields,
        "compare_result": compare_result,
        "comparison_text": comparison_text,
        "llm_final_report": llm_final_report,
        "saved_path": saved_path,
    }

In [16]:
# 실행 예시
result = create_report_comparison_result_v2(
    baseline_docx_path=baseline_path,
    compare_docx_path=compare_path,
    output_docx_path=output_compare_report,
    invoke_llm_fn=invoke_llm,
)

print(json.dumps(result["baseline_fields"], ensure_ascii=False, indent=2, default=str))
print()
print(result["comparison_text"])
print()
print(json.dumps(result["llm_final_report"], ensure_ascii=False, indent=2, default=str))

{
  "document_name": "battery_report_1000.docx",
  "generated_at": null,
  "source_file_name": null,
  "row_count": null,
  "voltage_max": null,
  "voltage_min": null,
  "temperature_max": null,
  "temperature_min": null,
  "module_status": null,
  "llm_summary": "이 배터리 충방전 시험 데이터 분석 레포트는 배터리의 용량, 용접, 센서 성능을 평가합니다. 전반적으로 배터리의 품질과 신뢰성을 평가하는 데 목적이 있습니다.",
  "llm_verdict": null,
  "voltage_range": null,
  "temperature_range": null
}

기준 문서: battery_report_1000.docx
비교 문서: battery_report_1001.docx
전체 변경 필드 수: 2
수치형 변경 수: 0
범주형 변경 수: 1
텍스트형 변경 수: 1

[상세 변경 내역]
- 필드: llm_verdict (categorical)
  기준값: None
  비교값: Pass

- 필드: llm_summary (text)
  기준값: 이 배터리 충방전 시험 데이터 분석 레포트는 배터리의 용량, 용접, 센서 성능을 평가합니다. 전반적으로 배터리의 품질과 신뢰성을 평가하는 데 목적이 있습니다.
  비교값: 본 배터리 충방전 시험 데이터 분석 보고서는 배터리의 용량, 용접, 센서 성능을 평가하며, 전반적인 성능이 양호한 것으로 판단됩니다. 용량 측정 결과는 설계 사양을 만족하고, 용접 상태 또한 안정적이며, 센서 데이터도 정상 범위 내에서 작동함을 확인했습니다. 원시 데이터 요약은 이러한 분석 결과의 근거를 제공합니다.


{
  "comparison_summary": "두 배터리 평가 보고서 간의 비교 결과, 'llm_verdict' 필드에서 'Non